# NASWOT и точность CIFAR-100 после независимого обучения

Используем CIFAR-100 и аугментации из `training/cifar100.ipynb`. Модель устроена как в MNIST-эксперименте: один расширяемый полносвязный слой принимает развёрнутое изображение и выдаёт 100 классов. Каждую архитектуру оцениваем при случайных весах, затем отдельно обучаем с нуля.

**Предлагаемые настройки для сравнения:** 3 расширения, 100 эпох, batch 128, SGD (lr 0.0125, momentum 0.9), cosine decay на 100 эпох, до 1000 новых связей и 2048 примеров для выбора связей. Порог 1e-5 и агрегация `variance` взяты из исходного CIFAR-100-ноутбука. Исходный эксперимент использует 700 эпох и до 4000 связей; здесь бюджет снижен для нескольких независимых обучений.

Как и в исходном CIFAR-100-ноутбуке, `val_loader` использует официальный тестовый набор. Поэтому лучшая точность по эпохам подходит для этого сравнения, но не является независимой тестовой оценкой.


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, TensorDataset, Subset
from torchvision import datasets, transforms
from torchvision.transforms import v2
from functools import partial
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'
    os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
    torch.set_deterministic_debug_mode(True)



################################
#     RESTART     RUNTIME      #
################################
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
from ssonn.train.train import *


In [ ]:
N = 3
NUM_EPOCHS = 100
BATCH_SIZE = 128
FINDER_EXAMPLES = 2048
NUM_WORKERS = 4 if torch.cuda.is_available() else 0
hyperparams = {
    "expansion_thresholds": {"fc_output": 1e-5},
    "max_new_edges_per_expansion": 1000,
    "learning_rate": 0.0125,
    "weight_decay": 0,
    "t_max": NUM_EPOCHS,
}


In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10),
    transforms.ToTensor(),
    v2.RandomErasing(p=0.5, scale=(0.05, 0.2), value="random"),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
    transforms.Lambda(lambda x: x.view(-1)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
    transforms.Lambda(lambda x: x.view(-1)),
])

train_dataset = datasets.CIFAR100(root="./data", train=True, download=True, transform=transform_train)
val_dataset = datasets.CIFAR100(root="./data", train=False, download=True, transform=transform_test)
finder_source = datasets.CIFAR100(root="./data", train=True, download=False, transform=transform_test)
finder_indices = torch.randperm(len(finder_source), generator=g)[:FINDER_EXAMPLES].tolist()
finder_dataset = Subset(finder_source, finder_indices)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=g,
                          num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
finder_loader = DataLoader(finder_dataset, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())


In [ ]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=3 * 32 * 32, num_classes=100):
        super().__init__()
        self.fc_output = nn.Linear(input_size, num_classes)

    def forward(self, x):
        return self.fc_output(x)


In [ ]:
import copy
import pandas as pd
import matplotlib.pyplot as plt
from itertools import islice
from torch import optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from ssonn.train.train import expand_model, train_one_epoch, eval_one_epoch, weights_init, apply_sparsity_mask

torch.manual_seed(SEED)
model = SimpleFCN()
architecture_model = convert_to_scaled_net(model, layers_to_convert=[model.fc_output], device=device)
architecture_model.fc_output.dropout.p = 0.3


## NASWOT до обучения

Оценка использует знаки входа в `LeakyReLU` расширенных слоёв; десять фиксированных батчей берутся из `val_loader`.


In [ ]:
@torch.inference_mode()
def naswot(model: nn.Module, x: torch.Tensor):
    K = torch.zeros((len(x), len(x)), dtype=torch.float32, device=x.device)
    n_activations = 0

    def count_patterns(_, inputs):
        nonlocal n_activations
        A = (inputs[0] > 0).flatten(1).float()
        K.add_(A @ A.T + (1 - A) @ (1 - A).T)
        n_activations += A.shape[1]

    hooks = [m.register_forward_pre_hook(count_patterns)
             for m in model.modules() if isinstance(m, (nn.ReLU, nn.LeakyReLU))]
    try:
        model.eval()(x)
    finally:
        for hook in hooks:
            hook.remove()

    _, logabsdet = torch.linalg.slogdet(K.cpu().double())
    return logabsdet.item(), n_activations


## Расширение и независимое обучение


In [ ]:
eval_batches = [x.to(device) for x, _ in islice(val_loader, 10)]
edge_finder = EdgeFinder(
    metric=AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    dataloader=finder_loader,
    device=device,
    aggregation_mode="variance",
    max_to_replace=hyperparams["max_new_edges_per_expansion"],
)
criterion = nn.CrossEntropyLoss()
score_rows, result_rows = [], []

for step in range(1, N + 1):
    expanded, new_edges = expand_model(architecture_model, edge_finder,
                                       hyperparams["expansion_thresholds"])
    if not expanded:
        raise RuntimeError(f"Расширение {step} не выбрало ни одной связи")

    # Обновить маски после изменения размера слоя; модель поиска остаётся необученной.
    torch.manual_seed(SEED)
    architecture_model.apply(weights_init)
    candidate = copy.deepcopy(architecture_model)
    torch.manual_seed(SEED)  # одинаковый seed для сравнения архитектур
    candidate.apply(weights_init)

    scores = []
    for batch, x in enumerate(eval_batches):
        score, n_activations = naswot(candidate, x)
        scores.append(score)
        score_rows.append(dict(expansions=step, batch=batch, score=score,
                               n_activations=n_activations))

    apply_sparsity_mask(candidate)
    g.manual_seed(SEED)
    optimizer = optim.SGD(candidate.parameters(), lr=hyperparams["learning_rate"],
                          momentum=0.9, weight_decay=hyperparams["weight_decay"])
    scheduler = CosineAnnealingLR(optimizer, T_max=hyperparams["t_max"])
    best_val_accuracy, best_epoch = -1.0, 0
    for epoch in range(1, NUM_EPOCHS + 1):
        train_one_epoch(candidate, train_loader, optimizer, criterion, device)
        _, val_accuracy = eval_one_epoch(candidate, criterion, val_loader,
                                         "classification", device)
        scheduler.step()
        if val_accuracy > best_val_accuracy:
            best_val_accuracy, best_epoch = val_accuracy, epoch
        if epoch % 10 == 0 or epoch == NUM_EPOCHS:
            print(f"{step} расширений | эпоха {epoch}/{NUM_EPOCHS} | val_accuracy={val_accuracy:.4f}")

    result_rows.append(dict(expansions=step, mean_naswot=np.mean(scores),
                            best_val_accuracy=best_val_accuracy,
                            best_epoch=best_epoch, new_edges=new_edges))
    del candidate, optimizer, scheduler

score_df = pd.DataFrame(score_rows)
results_df = pd.DataFrame(result_rows)
assert len(score_df) == 10 * N and len(results_df) == N
results_df


## Корреляция NASWOT и лучшей val_accuracy

Каждая точка обозначает одну архитектуру; её NASWOT усреднён по 10 батчам.


In [ ]:
x = results_df["mean_naswot"]
y = results_df["best_val_accuracy"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].scatter(x, y)
axes[0].set(xlabel="Средний NASWOT logdet", ylabel="Лучшая val_accuracy",
            title=f"Пирсон r = {x.corr(y):.3f}")

x_rank, y_rank = x.rank(), y.rank()
axes[1].scatter(x_rank, y_rank)
axes[1].set(xlabel="Ранг NASWOT", ylabel="Ранг val_accuracy",
            title=f"Спирмен ρ = {x.corr(y, method='spearman'):.3f}")

for ax, xs, ys in ((axes[0], x, y), (axes[1], x_rank, y_rank)):
    for step, xi, yi in zip(results_df["expansions"], xs, ys):
        ax.annotate(str(step), (xi, yi), xytext=(5, 5), textcoords="offset points")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
